# Deduplicación de Clientes - TABLA_MASTER_ID

## Objetivo
Identificar clientes duplicados en la base de datos usando un algoritmo de grafos basado en Email y Celular.

## Metodología
1. Cargar clientes desde SQL Server
2. Limpiar y normalizar datos (email, celular)
3. Validar calidad de datos
4. Crear grafo de conectividad (si comparten email o celular, están conectados)
5. Identificar componentes conectadas = clientes reales únicos
6. Asignar ID único a cada cliente real
7. Generar reporte con métricas de duplicados

## Output
- Archivo Excel con tabla de clientes deduplicados
- Reporte de estadísticas y duplicados encontrados

## 1. Importar Librerías

In [11]:
import pandas as pd
import re
from sqlalchemy import create_engine, text
import urllib
import networkx as nx
import os
from dotenv import load_dotenv
from datetime import datetime

# Cargar variables de entorno
load_dotenv()

# # Configurar opciones de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ Librerías importadas correctamente")

✓ Librerías importadas correctamente


## 2. Conexión a SQL Server

In [12]:
# CONEXIÓN A SQL SERVER

try:
    params = urllib.parse.quote_plus(
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={os.getenv('DB_SERVER')};"
        "DATABASE=Ventas_Comerssia;"
        f"UID={os.getenv('DB_USER')};"
        f"PWD={os.getenv('DB_PASSWORD')};"
    )

    engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
    
    # Validar conexión
    with engine.connect() as conn:
        result = conn.execute(text("SELECT @@version"))
        version = result.fetchone()[0]
    
    print("✓ Conexión a SQL Server exitosa")
    print(f"  Servidor: {os.getenv('DB_SERVER')}")
    print(f"  Base de datos: Ventas_Comerssia")

except Exception as e:
    print(f"✗ Error al conectar a SQL Server: {str(e)}")
    print("  Verifica las variables de entorno en .env")
    raise

✓ Conexión a SQL Server exitosa
  Servidor: 181.57.189.150,34276
  Base de datos: Ventas_Comerssia


## 3. Cargar y Explorar Datos

In [13]:
# CARGAR DATOS

query = """
SELECT 
    Cliente,
    Email,
    Celular,
    Nombres
FROM dbo.Clientes
"""

df = pd.read_sql(query, engine)

print(f"✓ Datos cargados")
print(f"  Registros totales: {len(df):,}")
print(f"\n📊 Primeras filas:")
print(df.head(10))

✓ Datos cargados
  Registros totales: 407,684

📊 Primeras filas:
        Cliente                         Email     Celular  \
0     C93239908      barrerajuliana@gmail.com               
1     C93398572  hector.godoy@unibague.edu.co  3208032231   
2     C98518264     juanpalacio9754@gmail.com  3154593854   
3   C1037662613       maluliderazgo@gmail.com  3053552703   
4    CC42960325       sicoastrologa@gmail.com  3014019107   
5     C71772403        camilitojara@gmail.com  1111111111   
6     C79795727        richietamayo@gmail.com  3118397628   
7    C901799835           kchakafit@gmail.com  3102560656   
8     C67030813      monica_londo@hotmail.com  3104217952   
9  CC1044421468           caromugno@gmail.com  3015587868   

                Nombres  
0           Juan Felipe  
1                HECTOR  
2                  JUAN  
3           MARÍA LUCÍA  
4               BEATRIZ  
5                CAMILO  
6               RICHARD  
7  THE FLOW DANCE S.A.S  
8                Monica  
9  

## 4. Análisis de Datos Faltantes

In [14]:
# ANÁLISIS DE VALORES FALTANTES

print("\n📋 Valores Faltantes:")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)

missing_df = pd.DataFrame({
    'Columna': df.columns,
    'Nulos': missing.values,
    '% Nulos': missing_pct.values
})

print(missing_df.to_string(index=False))

print("\n📊 Estadísticas por Columna:")
print(df.describe(include='all').to_string())


📋 Valores Faltantes:
Columna  Nulos  % Nulos
Cliente      0     0.00
  Email      0     0.00
Celular      0     0.00
Nombres   1090     0.27

📊 Estadísticas por Columna:
            Cliente                 Email     Celular Nombres
count        407684                407684      407684  406594
unique       407683                263149      288490   70854
top     C1098660341  negado@provenzal.net  1111111111   MARIA
freq              2                134865      102027    7585


## 5. Limpieza y Normalización de Datos

In [15]:
# LIMPIEZA DE DATOS

# Crear copia para trabajar
df_clean = df.copy()

# Lista de valores a ignorar (configurables)
INVALID_EMAILS = ['negado@provenzal.net', 'none', '', 'sin@email.com', 'test@test.com']
INVALID_PHONES = ['1111111111', 'nan', '', '0000000000', '9999999999']

# Valores centinela para conservar invalidos en output
SENTINEL_EMAIL = 'negado@provenzal.net'
SENTINEL_PHONE = '1111111111'

# Normalizar y limpiar EMAIL
df_clean['Email'] = df_clean['Email'].astype(str).str.lower().str.strip()
df_clean.loc[df_clean['Email'].isin(INVALID_EMAILS), 'Email'] = SENTINEL_EMAIL

# Validar formato de email basico y marcar invalido con centinela
df_clean.loc[~df_clean['Email'].str.contains('@', na=False), 'Email'] = SENTINEL_EMAIL

# Normalizar y limpiar CELULAR
df_clean['Celular'] = df_clean['Celular'].astype(str).str.strip()

# Mantener solo digitos
df_clean['Celular'] = df_clean['Celular'].apply(lambda x: re.sub(r'[^0-9]', '', str(x)) if x else '')
df_clean.loc[df_clean['Celular'].isin(INVALID_PHONES), 'Celular'] = SENTINEL_PHONE
df_clean.loc[df_clean['Celular'].str.len() != 10, 'Celular'] = SENTINEL_PHONE

# Normalizar NOMBRES
df_clean['Nombres'] = df_clean['Nombres'].astype(str).str.upper().str.strip()

print("✓ Datos limpios y normalizados")
print("\n📊 Cambios después de limpieza:")
emails_validos = (df_clean['Email'] != SENTINEL_EMAIL).sum()
celulares_validos = (df_clean['Celular'] != SENTINEL_PHONE).sum()
print(f"  Emails válidos: {emails_validos:,} ({emails_validos/len(df_clean)*100:.1f}%)")
print(f"  Celulares válidos: {celulares_validos:,} ({celulares_validos/len(df_clean)*100:.1f}%)")


print("\n📊 Estadísticas por Columna (datos limpios):")
print(df_clean.describe(include='all').to_string())


✓ Datos limpios y normalizados

📊 Cambios después de limpieza:
  Emails válidos: 272,499 (66.8%)
  Celulares válidos: 304,682 (74.7%)

📊 Estadísticas por Columna (datos limpios):
            Cliente                 Email     Celular Nombres
count        407684                407684      407684  406594
unique       407683                263144      288218   67749
top     C1098660341  negado@provenzal.net  1111111111   MARIA
freq              2                135185      103002    7807


## 6. Crear Grafo de Conectividad

In [16]:
# CREAR GRAFO DE CONECTIVIDAD

G = nx.Graph()

# Solo usar datos validos para unir en el grafo
email_valid_mask = df_clean['Email'].notna() & (df_clean['Email'] != SENTINEL_EMAIL)
cel_valid_mask = df_clean['Celular'].notna() & (df_clean['Celular'] != SENTINEL_PHONE)

# Agregar nodos (todos los clientes)
for cliente in df_clean['Cliente']:
    G.add_node(cliente)

# Agrupar por EMAIL y crear bordes (prioridad principal)
email_groups = df_clean[email_valid_mask].groupby('Email')['Cliente'].apply(list)
email_duplicados = 0

for email, clientes in email_groups.items():
    if len(clientes) > 1:
        email_duplicados += len(clientes) - 1
        # Conectar clientes con el mismo email (cadena)
        for i in range(len(clientes) - 1):
            G.add_edge(clientes[i], clientes[i + 1])

# Frecuencia global de email valido para validar unicidad global
audit_email_freq = df_clean[email_valid_mask].groupby('Email').size()

# Agrupar por CELULAR y crear bordes con regla hibrida
# - Solo celulares validos (distintos del centinela)
# - Si el email es sentinela o unico globalmente, se pueden unir por celular
cel_groups = df_clean[cel_valid_mask].groupby('Celular')
cel_duplicados = 0
cel_uniones_email_unico = 0
cel_grupos_email_unico = 0
cel_grupos_con_sentinel = 0

for cel, group in cel_groups:
    if len(group) <= 1:
        continue

    # Unir subgrupos con el mismo email valido dentro del celular
    group_email_valido = group[group['Email'] != SENTINEL_EMAIL]
    for _, sub in group_email_valido.groupby('Email'):
        sub_clientes = sub['Cliente'].tolist()
        if len(sub_clientes) > 1:
            cel_duplicados += len(sub_clientes) - 1
            for i in range(len(sub_clientes) - 1):
                G.add_edge(sub_clientes[i], sub_clientes[i + 1])

    # Unir por celular cuando:
    # - email sentinela (invalido estandarizado), o
    # - email valido unico globalmente
    mask_union_cel = (
        group['Email'].eq(SENTINEL_EMAIL)
        | (
            (group['Email'] != SENTINEL_EMAIL)
            & group['Email'].map(audit_email_freq).eq(1)
        )
    )
    clientes_union_cel = group.loc[mask_union_cel, 'Cliente'].tolist()

    if len(clientes_union_cel) > 1:
        cel_grupos_email_unico += 1
        cel_uniones_email_unico += len(clientes_union_cel) - 1
        cel_duplicados += len(clientes_union_cel) - 1

        if group['Email'].eq(SENTINEL_EMAIL).any():
            cel_grupos_con_sentinel += 1

        for i in range(len(clientes_union_cel) - 1):
            G.add_edge(clientes_union_cel[i], clientes_union_cel[i + 1])

print("✓ Grafo creado")
print(f"  Nodos: {G.number_of_nodes():,}")
print(f"  Bordes (email): {email_duplicados:,}")
print(f"  Bordes (celular): {cel_duplicados:,}")
print(f"  Uniones por celular (email sentinela o unico global): {cel_uniones_email_unico:,}")
print(f"  Grupos de celular que activaron esa regla: {cel_grupos_email_unico:,}")
print(f"  Grupos de celular con al menos un email sentinela: {cel_grupos_con_sentinel:,}")

✓ Grafo creado
  Nodos: 407,683
  Bordes (email): 9,356
  Bordes (celular): 15,414
  Uniones por celular (email sentinela o unico global): 9,923
  Grupos de celular que activaron esa regla: 8,550
  Grupos de celular con al menos un email sentinela: 3,880


## 7. Identificar Clientes Únicos Reales

In [17]:
# CLUSTERS = CLIENTES REALES ÚNICOS

componentes = list(nx.connected_components(G))
clientes_unicos_reales = len(componentes)

# Crear mapeo de cliente original a ID único
map_cliente = {}

for i, comp in enumerate(componentes):
    id_unico = 100000 + i  # IDs desde 100000
    for cliente in comp:
        map_cliente[cliente] = id_unico

# Agregar ID único al dataframe
df_clean['ID_Cliente_Unico'] = df_clean['Cliente'].map(map_cliente)
df_clean['Es_Duplicado'] = df_clean.groupby('ID_Cliente_Unico')['Cliente'].transform('count') > 1

print("✓ Clientes únicos reales identificados")
print(f"  Clientes reales únicos: {clientes_unicos_reales:,}")
print(f"  Duplicados encontrados: {df_clean['Es_Duplicado'].sum():,}")
print(f"  Tasa de deduplicación: {(1 - clientes_unicos_reales / len(df_clean)) * 100:.2f}%")

✓ Clientes únicos reales identificados
  Clientes reales únicos: 388,405
  Duplicados encontrados: 36,244
  Tasa de deduplicación: 4.73%


## 8. Análisis de Duplicados

In [18]:
# ANÁLISIS DETALLADO DE DUPLICADOS

total_registros = len(df_clean)
total_duplicados = df_clean['Es_Duplicado'].sum()

print("\n📊 Estadísticas de Duplicados:")
print(f"  Total de registros:       {total_registros:,}")
print(f"  Clientes únicos reales:   {clientes_unicos_reales:,}")
print(f"  Registros duplicados:     {total_duplicados:,} ({total_duplicados / total_registros * 100:.2f}% del total)")

print(f"\n📐 Distribución por tamaño de grupo:")
print(f"  {'Tamaño':>8} {'Grupos':>10} {'Registros':>12} {'% del total':>13}")
print(f"  {'-------':>8} {'-------':>10} {'---------':>12} {'-----------':>13}")

group_sizes = df_clean.groupby('ID_Cliente_Unico').size()
size_dist = group_sizes.value_counts().sort_index()

for size, count in size_dist.items():
    registros = count * size
    pct = registros / total_registros * 100
    print(f"  {size:>8} {count:>10,} {registros:>12,} {pct:>12.2f}%")



📊 Estadísticas de Duplicados:
  Total de registros:       407,684
  Clientes únicos reales:   388,405
  Registros duplicados:     36,244 (8.89% del total)

📐 Distribución por tamaño de grupo:
    Tamaño     Grupos    Registros   % del total
   -------    -------    ---------   -----------
         1    371,440      371,440        91.11%
         2     15,533       31,066         7.62%
         3      1,045        3,135         0.77%
         4        196          784         0.19%
         5         88          440         0.11%
         6         44          264         0.06%
         7         30          210         0.05%
         8         16          128         0.03%
         9          6           54         0.01%
        10          3           30         0.01%
        13          1           13         0.00%
        24          1           24         0.01%
        27          1           27         0.01%
        69          1           69         0.02%


## 9. Reordenar Columnas y Preparar Output

In [19]:
# REORDENAR COLUMNAS Y PREPARAR PARA EXPORTAR

df_output = df_clean[['ID_Cliente_Unico', 'Cliente', 'Nombres', 'Email', 'Celular', 'Es_Duplicado']].copy()
df_output = df_output.sort_values(['ID_Cliente_Unico', 'Cliente'])

print("✓ Datos preparados para exportar")
print(f"\n📋 Vista final de datos:")
print(df_output.head(20).to_string())

✓ Datos preparados para exportar

📋 Vista final de datos:
        ID_Cliente_Unico       Cliente               Nombres                         Email     Celular  Es_Duplicado
0                 100000     C93239908           JUAN FELIPE      barrerajuliana@gmail.com  1111111111         False
1                 100001     C93398572                HECTOR  hector.godoy@unibague.edu.co  3208032231          True
400426            100001     C93398576        HECTOR ERLENDI            godoy.he@gmail.com  3208032231          True
2                 100002     C98518264                  JUAN     juanpalacio9754@gmail.com  3154593854         False
3                 100003   C1037662613           MARÍA LUCÍA       maluliderazgo@gmail.com  3053552703         False
147354            100004     C21335221                ANGELA       sicoastrologa@gmail.com  3014019107          True
239769            100004     C42960325               BEATRIZ       sicoastrologa@gmail.com  1111111111          True
4     

## 10. Exportar Resultados

In [20]:
# EXPORTAR A EXCEL Y SQL CON ESTRUCTURA

timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
output_file = "Clientes_Unicos.xlsx"

# Tabla de salida en SQL
sql_table_clientes = "MASTER_ID"

try:
    # Hoja 2: Resumen estadistico (solo para Excel)
    resumen = pd.DataFrame({
        'Métrica': [
            'Fecha de ejecución',
            'Registros originales',
            'Clientes únicos reales',
            'Duplicados encontrados',
            'Tasa de deduplicación',
            'Emails válidos',
            'Celulares válidos',
            'Uniones por celular (email sentinela o único global)',
            'Grupos celular auditados por esa regla',
            'Grupos con email sentinela en regla celular'
        ],
        'Valor': [
            timestamp,
            len(df_clean),
            clientes_unicos_reales,
            df_clean['Es_Duplicado'].sum(),
            f"{(1 - clientes_unicos_reales / len(df_clean)) * 100:.2f}%",
            (df_clean['Email'] != SENTINEL_EMAIL).sum(),
            (df_clean['Celular'] != SENTINEL_PHONE).sum(),
            cel_uniones_email_unico,
            cel_grupos_email_unico,
            cel_grupos_con_sentinel
        ]
    })

    # 1) Exportar a Excel
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        df_output.to_excel(writer, sheet_name='Clientes_Deduplicados', index=False)
        resumen.to_excel(writer, sheet_name='Resumen', index=False)

    # 2) Exportar a SQL (solo tabla de IDs unicos)
    df_output.to_sql(sql_table_clientes, con=engine, if_exists='replace', index=False)

    print(f"✓ Archivo exportado: {output_file}")
    print("  Hojas: 'Clientes_Deduplicados', 'Resumen'")
    print(f"✓ Tabla SQL exportada: {sql_table_clientes}")

except Exception as e:
    print(f"✗ Error al exportar: {str(e)}")

✓ Archivo exportado: Clientes_Unicos.xlsx
  Hojas: 'Clientes_Deduplicados', 'Resumen'
✓ Tabla SQL exportada: MASTER_ID


## 11. Resumen Final

In [21]:
# RESUMEN FINAL

print("\n" + "="*60)
print("RESUMEN DE DEDUPLICACIÓN DE CLIENTES")
print("="*60)
print(f"\n📅 Fecha y hora: {timestamp}")
print(f"\n📊 RESULTADOS:")
print(f"  • Registros procesados: {len(df_clean):,}")
print(f"  • Clientes únicos reales: {clientes_unicos_reales:,}")
print(f"  • Duplicados identificados: {df_clean['Es_Duplicado'].sum():,}")
print(f"  • Tasa de deduplicación: {(1 - clientes_unicos_reales / len(df_clean)) * 100:.2f}%")
print(f"\n🔗 CONEXIÓN POR ATRIBUTOS:")
print(f"  • Clientes con email válido: {(df_clean['Email'] != SENTINEL_EMAIL).sum():,}")
print(f"  • Clientes con celular válido: {(df_clean['Celular'] != SENTINEL_PHONE).sum():,}")
print(f"  • Uniones por celular (email sentinela o único global): {cel_uniones_email_unico:,}")
print(f"  • Grupos celular auditados por esa regla: {cel_grupos_email_unico:,}")
print(f"  • Grupos con email sentinela en regla celular: {cel_grupos_con_sentinel:,}")
print(f"\n💾 EXPORTACIÓN:")
print(f"  • Archivo: {output_file}")
print(f"  • Registros exportados: {len(df_output):,}")
print(f"  • Formato: Excel (.xlsx)")
print("\n" + "="*60)



RESUMEN DE DEDUPLICACIÓN DE CLIENTES

📅 Fecha y hora: 2026-04-20 08:20:09

📊 RESULTADOS:
  • Registros procesados: 407,684
  • Clientes únicos reales: 388,405
  • Duplicados identificados: 36,244
  • Tasa de deduplicación: 4.73%

🔗 CONEXIÓN POR ATRIBUTOS:
  • Clientes con email válido: 272,499
  • Clientes con celular válido: 304,682
  • Uniones por celular (email sentinela o único global): 9,923
  • Grupos celular auditados por esa regla: 8,550
  • Grupos con email sentinela en regla celular: 3,880

💾 EXPORTACIÓN:
  • Archivo: Clientes_Unicos.xlsx
  • Registros exportados: 407,684
  • Formato: Excel (.xlsx)

